# Implementation — Trading Costs, Leverage, Shorting
## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Estimate transaction costs** (spread, market impact) for a strategy
2. **Compute implementation shortfall** — paper return vs realized return
3. **Apply leverage** correctly, accounting for borrowing cost
4. **Understand the mechanics of shorting** — borrow cost, hard-to-borrow names
5. **Audit AI-generated backtests** for naive cost handling

## 📋 TOC
1. [Setup](#setup)  2. [Why Implementation Matters](#why)
3. [Pitfall Checklist](#pitfalls)  4. [Transaction Costs](#costs)
5. [Implementation Shortfall](#shortfall)  6. [Leverage and Borrowing](#leverage)
7. [Shorting](#shorting)  8. [🎯 Challenge: Net Sharpe](#challenge)
9. [Submission](#submit)  10. [Key Takeaways](#takeaways)

---
## 🛠️ Setup <a id="setup"></a>

In [ ]:
#@title Setup
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize']=[10,5]; plt.rcParams['font.size']=11
import warnings; warnings.filterwarnings('ignore')
print("✅ Loaded")

---
## Why Implementation Matters <a id="why"></a>

Backtests show gross returns. Real returns are net of:
- **Spread** — half the bid-ask, paid every time you trade
- **Market impact** — the price moves against you as you trade
- **Borrowing cost** for leverage
- **Borrowing cost for short positions** (sometimes negative — you earn rebate on the cash)
- **Financing** for futures/forwards
- **Taxes** (we mostly ignore in academic work; matters for taxable accounts)

**A strategy with 4% gross return and 3% total cost has only 1% net.** This is
the difference between paper alpha and tradeable alpha.

---
## 🛡️ Pitfall Checklist <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---------|-----------------|-------------------|
| 1 | **Backtesting at close prices** | Real fills happen at midpoint or worse; backtests overstate returns by ~5bps/trade | Use midpoint or apply explicit spread cost |
| 2 | **Ignoring market impact** | A $10M trade in a small-cap moves the price 50bp+; backtests assume infinite liquidity | Estimate impact using ADV (avg daily volume) |
| 3 | **Quoting Sharpe without specifying capacity** | A strategy works on $10M may not work on $1B | Always report Sharpe at multiple capital levels |
| 4 | **Forgetting financing in long-short** | Long $X, short $X has zero net capital — but you pay financing on $X | Long-short P&L net of financing is what matters |
| 5 | **Hard-to-borrow names** | Some short positions cost 5-20% to maintain | Always check borrow rate before backtesting a short |

---
## Transaction Costs <a id="costs"></a>

Simplest model: **cost ≈ half_spread × turnover**

Large-cap stocks (SPY, AAPL): half-spread ≈ 1 bp
Mid-cap: 3-10 bp
Small-cap: 10-50 bp
Emerging markets: 30-100 bp

For high-turnover strategies (momentum, statistical arbitrage), this is the
dominant cost item.

**Market impact** adds non-linearly with trade size:

$$\text{Impact} \approx \eta \cdot \sigma \cdot \sqrt{Q / ADV}$$

where Q is trade size, ADV is average daily volume, η ≈ 0.1-0.5 (estimated).

For trades < 1% of ADV, impact is small. For trades > 10% of ADV, it dominates.

---
## Implementation Shortfall <a id="shortfall"></a>

The **implementation shortfall** measures the gap between paper P&L and realized P&L:

$$IS = P_{\text{decision}} \cdot Q - P_{\text{fill}} \cdot Q$$

You wanted to buy 1000 shares of AAPL when it was $150. By the time you
finished filling, the average price was $150.20. IS = $200 — that's your
cost.

This is the **single most-watched metric** at execution desks. Algorithm
performance is graded on IS reduction.

---
## Leverage and Borrowing <a id="leverage"></a>

Leverage = (Long + Short Notional) / NAV. A market-neutral fund with $100M
long and $100M short has 2x leverage, 0 net market exposure.

You borrow to fund leverage. Borrow rate depends:
- Broker: typically O/N rate + 50-100 bp
- Repo: O/N rate + 10-30 bp (for sov bonds, etc.)
- Securities lending: variable

**Net excess return** = (gross strategy return × leverage) − (financing cost × leverage − 1)

A 4% gross strategy at 2x leverage with 6% financing cost on the second
dollar = 4×2 − 6×1 = 2% net excess. **Higher leverage isn't always better.**

---
## Shorting <a id="shorting"></a>

To short a stock, you:
1. Borrow it from a long holder (via your prime broker)
2. Sell it
3. Receive cash → put it in an account (earns short rebate)
4. Eventually buy it back to return the borrow

**Costs:**
- **Borrow fee**: 25 bp/yr typical for liquid names, can spike to 50-100%+ for
  hard-to-borrow (GME during 2021)
- **Recall risk**: if the lender wants their shares back, you must close

**Most quant strategies require shorting.** Long-only versions exist but
typically capture half the premium and double the risk.

---
## 🎯 Challenge: Net Sharpe <a id="challenge"></a>

> **Setup.** Your momentum strategy has these backtest stats:
> - Gross annual return: 8%
> - Annual volatility: 12%
> - Annual turnover: 150%
> - You'll deploy with 2x leverage. Financing cost: 6% per year on excess capital.
> - Trading cost: 8 bp per turn (round-trip).
> - 30% of positions are short. Average borrow cost: 50 bp/yr on shorts.

### Q1 — Trading cost drag

> **📌 Required:**
> ```python
> turnover    = 1.50
> cost_per_turn_bp = 8
> trading_cost_drag = ____   # turnover * cost_per_turn_bp / 10000
> ```

In [ ]:
turnover = 1.50
cost_per_turn_bp = 8

trading_cost_drag = ____
print(f"Trading cost drag: {trading_cost_drag:.2%}")

### Q2 — Financing drag

You're at 2x leverage. You borrow 1x at 6%.

> **📌 Required:**
> ```python
> leverage     = 2
> financing_cost = 0.06
> financing_drag = ____   # (leverage - 1) * financing_cost
> ```

In [ ]:
leverage = 2
financing_cost = 0.06

financing_drag = ____
print(f"Financing drag: {financing_drag:.2%}")

### Q3 — Short borrow drag

30% of leveraged book is short, paying 50bp on the short notional.

> **📌 Required:**
> ```python
> short_fraction  = 0.30
> short_borrow_bp = 50
> short_drag      = ____   # leverage * short_fraction * short_borrow_bp / 10000
> ```

In [ ]:
short_fraction  = 0.30
short_borrow_bp = 50

short_drag = ____
print(f"Short-borrow drag: {short_drag:.2%}")

### Q4 — Net Sharpe

Total drag = trading + financing + short. Net annual return = leveraged
gross − total drag. Net Sharpe = net return / leveraged vol.

> **📌 Required:**
> ```python
> gross_return = 0.08
> annual_vol   = 0.12
> leveraged_gross = leverage * gross_return
> leveraged_vol   = leverage * annual_vol
> total_drag      = trading_cost_drag + financing_drag + short_drag
> net_return      = ____
> net_sharpe      = ____
> ```

In [ ]:
gross_return = 0.08
annual_vol   = 0.12

leveraged_gross = leverage * gross_return
leveraged_vol   = leverage * annual_vol
total_drag      = trading_cost_drag + financing_drag + short_drag

net_return = ____
net_sharpe = ____

print(f"Gross Sharpe (unlevered):  {gross_return / annual_vol:.2f}")
print(f"Total drag:                 {total_drag:.2%}")
print(f"Net return (post-cost):     {net_return:.2%}")
print(f"Net Sharpe:                 {net_sharpe:.2f}")

### Q5 — Memo

Max 5 sentences. Decide whether to deploy. Cite (i) the gross vs net Sharpe,
(ii) the single biggest cost item, (iii) what could change your mind (e.g.,
lower turnover via better signal design, lower financing cost).

In [ ]:
MEMO = """Write your memo here."""
print(MEMO)

---
## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL ===
import json, base64, hashlib, datetime as dt
required = ["trading_cost_drag", "financing_drag", "short_drag", "net_return", "net_sharpe", "MEMO"]
missing = [v for v in required if v not in dir()]
if missing: raise NameError(f"\n❌ Missing: {missing}")
payload = {"assignment": "Implementation_AI",
    "ts": dt.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip()}
blob = json.dumps(payload, sort_keys=True)
token = f"UG54::{hashlib.sha256(blob.encode()).hexdigest()[:8]}::{base64.b64encode(blob.encode()).decode()}"
print("="*72); print(token); print("="*72)

---
## 🧠 Key Takeaways <a id="takeaways"></a>
1. **Net = Gross − Costs.** Always report both.
2. **Trading cost ≈ turnover × spread.** A 150%-turnover strategy at 10bp loses 15bp/year base.
3. **Leverage isn't free.** Financing cost on the borrowed dollar eats into the return.
4. **Shorting is asymmetric.** Most names are cheap to short; some (meme stocks) are catastrophically expensive.
5. **AI's backtest will quote gross numbers. You must layer in realistic costs before deploying.**